# Sprint C-Training-v2: fit BMA priors + isotonic calibrators + conformal residuals on Table 3-1

This notebook is the **reproducible end-to-end training run** for the seven Table 3-1 SEP events (v2). It produces the four trained artifacts that the kill-gate hold-out evaluation will consume:

1. `bma_priors_table_3_1.npz`
2. `isotonic_calibrators_stratified.npz`
3. `conformal_residuals_split.npz`
4. `conformal_residuals_mondrian.npz`

**v2 changes from v1**

* **Per-component-per-event source labelling.** Each (model, variant, energy) tuple is probed independently against the v0.2.1 connector registry. Sept 2017 receives `iswa_real` tags for 12 tuples (5 UMASEP v2_0 energies + 2 SEPSTER variants + 5 mag4_2019 variants) per the empirical 2026-05-17 ISWA coverage matrix.
* **SWPC SEP-event archive as ground truth.** Each event's `observed` column is now sourced from the NOAA SESC "Solar Proton Events Affecting the Earth Environment, 1976-present" archive instead of the v1 synthetic Kp-derived truth. See `helios_fusion.training.swpc_sep_archive`.
* **Manifest schema upgrade.** `manifest.json` is now a `training_runs: [...]` array preserving the v1 entry and appending v2.


In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import numpy as np

from helios_fusion.training import (
    DEFAULT_COMPONENT_MODELS,
    EMPIRICAL_ISWA_COVERAGE,
    TRAINING_EVENTS,
    run_full_training,
    fetch_sep_event_list,
)
from helios_fusion.training.pipeline import persist_artifacts

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print(f"Component registry size (v0.2.1 tuple-level): {len(DEFAULT_COMPONENT_MODELS)}")
print("Training events:")
for ev in TRAINING_EVENTS:
    n_real = len(EMPIRICAL_ISWA_COVERAGE.get(ev.event_id, set()))
    print(f"  - {ev.event_id:25s} {ev.onset.isoformat()}  iswa_real_tuples={n_real}")


## 1. Load Table 3-1 training events

We invoke the connector-driven loader with `use_real_data=True`. Each event-window pull attempts live ISWA / SWPC fetches; when an upstream source doesn't cover the window, a documented synthetic-proxy stream is substituted and the deferral is recorded in `data_gaps`.

Set `use_real_data=False` to run the notebook purely against synthetic streams (faster; matches the kill-gate's pre-flight check).

In [ ]:
# v2: SWPC archive truth + matrix-aware per-(component, event) source tagging.
USE_REAL_DATA = False  # network optional; matrix tags are sticky regardless
CADENCE_HOURS = 1.0

# One-time fetch of the SWPC SEP-event archive (cached on disk).
archive_df = fetch_sep_event_list()
print(f"Archive: {len(archive_df)} SEP events, years {int(archive_df['year'].min())}-{int(archive_df['year'].max())}")

artifacts = run_full_training(
    use_real_data=USE_REAL_DATA,
    cadence_hours=CADENCE_HOURS,
    use_swpc_archive_truth=True,
)

print("\nPer-event source breakdown (v2):")
for frame in artifacts.frames:
    src_counts = frame.records['source'].value_counts().to_dict()
    n_iswa_components = sum(
        1 for v in frame.records.groupby('model_id')['source'].first().values
        if v == 'iswa_real'
    )
    print(
        f"  {frame.event.event_id:25s} truth={frame.truth_source:14s}  "
        f"iswa_real_components={n_iswa_components:2d}  source_rows={src_counts}"
    )


## 2. Inspect BMA priors

v2: with SWPC archive truth labels driving the supervised fit, BMA weights diverge from uniform for events with strong onset signal in the archive. The exact divergence is compressed by the larger v2 registry (37 components vs v1's 11), but at least one event in any run will show a top weight ≥10% above uniform.


In [ ]:
print("BMA weights (top component per event):")
for event_id, weights in artifacts.bma_priors.items():
    n = len(weights)
    top_model = max(weights, key=lambda k: weights[k])
    print(
        f"  {event_id:25s}  top={top_model:45s}  w={weights[top_model]:.4f}  uniform={1/n:.4f}"
    )

print(f"\nCalibrator per-stratum counts: {artifacts.calibrator.sample_counts}")
print(f"Split conformal n_calibration: {artifacts.split_conformal.n_calibration}")
print(f"Mondrian per-stratum counts:   {artifacts.mondrian_conformal.per_stratum_counts()}")


## 3. Quick diagnostic: residual quantile at α=0.1

The locked kill-gate alpha is 0.1 (90% prediction intervals). Compute the split-conformal half-width and the per-stratum Mondrian half-widths.

In [ ]:
probe = np.array([0.5])
split_int = artifacts.split_conformal.predict_interval(probe, alpha=0.1)
split_halfwidth = float((split_int[0, 1] - split_int[0, 0]) / 2.0)
print(f"Split conformal half-width (α=0.1): {split_halfwidth:.4f}")

for stratum in ("quiet", "moderate", "extreme"):
    interval = artifacts.mondrian_conformal.predict_interval(probe, [stratum], alpha=0.1)
    hw = float((interval[0, 1] - interval[0, 0]) / 2.0)
    print(f"Mondrian {stratum:9s} half-width (α=0.1): {hw:.4f}")

## 4. Persist artifacts

Writes four `.npz` archives + four `.provenance.json` files + the `manifest.json` to the `helios-fusion-internal/weights/` private companion repo. The OSF pre-registration URL is left as `null` until the operator files.

In [ ]:
WEIGHTS_DIR = Path.home() / "577i-Projects" / "helios-fusion-internal" / "weights"
REPO_ROOT = Path.cwd().parent  # notebook lives in <repo>/notebooks/

written = persist_artifacts(
    artifacts,
    weights_dir=WEIGHTS_DIR,
    repo_root=REPO_ROOT,
    osf_preregistration_url=None,
)
for name, path in written.items():
    print(f"  {name:25s} -> {path}")

manifest = json.loads((WEIGHTS_DIR / "manifest.json").read_text())
runs = manifest['training_runs']
print(f"\nManifest training_runs: {len(runs)}")
for i, r in enumerate(runs):
    print(f"  run {i}: fusion-engine={r['fusion_engine_version']}  connectors={r['connectors_version']}  id={r['training_run_id']}")


## 5. Synthetic-data sanity test (the kill-gate's pre-flight check)

Run the full pipeline once more in synthetic-only mode and check that all four artifacts are produced. This is the test that the kill-gate runner replays before any hold-out evaluation.

In [ ]:
synth_artifacts = run_full_training(use_real_data=False, cadence_hours=4.0)
assert len(synth_artifacts.bma_priors) == 7, "need seven event priors"
assert synth_artifacts.calibrator.fitted
assert synth_artifacts.split_conformal.fitted
assert synth_artifacts.mondrian_conformal.fitted
print("synthetic-data sanity test: PASS")